# An Unintelligent Attempt: Teaching Greedy Workers to Read the Market
---

"Wow I can make an AI myself, let's make an intelligent system!" I thought I was being smart. Then all this calculated best-path deterministic agents destroyed everything. Every day I saw my code get more and more convoluted, only for all that progress regressed, all because one change in a decision jumbled all the priorities for the rest of the game. And that's the first version, a greedy priority first system, where the worker would scramble to finish all the high priority tasks altogether...

and travelled all around the map while doing it.

## The Rewrite

First of all... Welcome to my first notebook on Kaggle. After that horrible first attempt on an agent, I decided to go back to the drawing board and do a full rewrite of the worker system. The result: a greedy queue-chaining system, designed to claim all the possible tile that it could work on, without having workers travel all across the map.

I wanted to answer one question: can better task execution and market-aware planting close the gap against a strong deterministic agent? With this rewrite, I had to find a new benchmark for it. And so *Read the Market, Choose the Farm* by Indar Karhana, a deterministic agent that switches strategies using early signals, became the local benchmark.

### Evaluation setup

I tested each version against *Read the Market, Choose the Farm* using
kaggle-environments 1.32.7. The era comparison uses three seeds from both
seats, giving six games per version. I measure final-money margin against the
same opponent.

The Plot Mixer sections use two selected traces: one vegetable-heavy game and
one animal-heavy game. They show how the agent behaves under different shop
rolls; they do not estimate its average behavior.

## The Plot Mixer

The worker rewrite improved performance, but it still wasn't good enough. So, naturally, the next thing to attack was the market recipe.

A fixed recipe ignores the market and how can this be solved? I decided to predict the value of crops over the next few days. Afterwards, my agent re-weights its crop mix every day by expected profit: share × sqrt(projected_value / average_value). The chart below shows the recipe (base) vs what the formula actually planned over a real game, and vs what ended up in the ground.

I'll show two games: one game where vegetable demands was high (Pet Cafes, Farmers Market), and another where animal produce demand was high (Yarn Store, Ice Cream Shop)

In [ ]:
import kaggle_environments as ke
from kaggle_environments import make

print("kaggle-environments version:", ke.__version__)
env = make("kaggriculture", configuration={"episodeSteps": 720}, debug=True)
print("env ready:", env.name)

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt

with open("/kaggle/input/datasets/rakhansyah/kaggriculture-trace-results/trace_veg.json") as f:
    trace = json.load(f)

BASE_MIX = {"STRAWBERRY": 0.6, "WHEAT": 0.1, "CARROT": 0.15, "TOMATO": 0.15}
MIX_CROPS = list(BASE_MIX.keys())
PLANT_CROPS = MIX_CROPS + ["MELON"]

rows = []
for gi, g in enumerate(trace["games"]):
    for d in g["per_day"]:
        rows.append({
            "game": gi, "day": d["day"],
            **{f"mix_{c}": d["dynamic_mix"].get(c, 0.0) for c in MIX_CROPS},
            **{f"planted_{c}": d["planted_counts"].get(c, 0) for c in PLANT_CROPS},
        })
df = pd.DataFrame(rows)

avg_dyn = df[df.game == 0][[f"mix_{c}" for c in MIX_CROPS]].mean()
x = range(len(MIX_CROPS)); w = 0.38
plt.figure(figsize=(9, 5))
plt.bar([i - w/2 for i in x], [BASE_MIX[c] for c in MIX_CROPS], w, label="base recipe")
plt.bar([i + w/2 for i in x], [avg_dyn[f"mix_{c}"] for c in MIX_CROPS], w,
        label="dynamic mix (avg planned)")
plt.xticks(list(x), MIX_CROPS); plt.ylabel("share of plantable tiles")
plt.legend(); plt.title("Recipe vs what the mix formula plans")
plt.tight_layout()

g0 = df[df.game == 0].set_index("day")
cols = [f"planted_{c}" for c in PLANT_CROPS]
frac = g0[cols].div(g0[cols].sum(axis=1), axis=0).fillna(0)
frac.plot.area(stacked=True, figsize=(9, 5),
               title="Realized crop mix over days (Hinge heavy game)")
plt.ylabel("share of occupied tiles"); plt.xlabel("day")
plt.tight_layout()

The mix formula re-weights every crop every day by its projected value, and you can see the market pulling it around. This game rolled heavy vegetable shops all-around, and you can see the planned share of carrots jumped from 15% → ~35%. The realized crop mix is also shown.

Early: the melon rush (11-12 tiles of them, melon is the best profit-per-tile-day in the game, so it's hard-capped), A signal also fires on melon harvest day, telling the agent to harvest and sell first. The goal is to front-run the opponent so they are the ones forced to sell into the lower price. 

Mid-Game: strawberry starts to surge as its prices rises above $200 from the local demand. While their share in the mix quickly drops in the next 2-3 days, because of its surge in profitability the planner already bought so much seeds for it, and due to the long lifetime of the plant, other plants cannot realize their share. 

Late-game: carrot and tomato start to surge as their prices rocket, and by day 29 the field is mostly carrot, scraping endgame coins due to its quick cycle.

In [ ]:

with open("/kaggle/input/datasets/rakhansyah/kaggriculture-trace-results/trace_ani.json") as f:
    trace = json.load(f)

rows = []
for gi, g in enumerate(trace["games"]):
    for d in g["per_day"]:
        rows.append({
            "game": gi, "day": d["day"],
            **{f"mix_{c}": d["dynamic_mix"].get(c, 0.0) for c in MIX_CROPS},
            **{f"planted_{c}": d["planted_counts"].get(c, 0) for c in PLANT_CROPS},
        })
df = pd.DataFrame(rows)

avg_dyn = df[df.game == 0][[f"mix_{c}" for c in MIX_CROPS]].mean()
x = range(len(MIX_CROPS)); w = 0.38
plt.figure(figsize=(9, 5))
plt.bar([i - w/2 for i in x], [BASE_MIX[c] for c in MIX_CROPS], w, label="base recipe")
plt.bar([i + w/2 for i in x], [avg_dyn[f"mix_{c}"] for c in MIX_CROPS], w,
        label="dynamic mix (avg planned)")
plt.xticks(list(x), MIX_CROPS); plt.ylabel("share of plantable tiles")
plt.legend(); plt.title("Recipe vs what the mix formula plans")
plt.tight_layout()

g0 = df[df.game == 0].set_index("day")
cols = [f"planted_{c}" for c in PLANT_CROPS]
frac = g0[cols].div(g0[cols].sum(axis=1), axis=0).fillna(0)
frac.plot.area(stacked=True, figsize=(9, 5),
               title="Realized crop mix over days (Animal heavy game)")
plt.ylabel("share of occupied tiles"); plt.xlabel("day")
plt.tight_layout()

This game rolled more animal produce shops (Yarn Store, Ice Cream Shop), so the mix barely moved: strawberry 60% to 48%, vegetables in single digits. The endgame flips hard to wheat, to feed all the animals now producing wool and milk.

## The Hinge Update

The new engine update (1.32.7) made carrots and tomatoes more viable especially late game. Their prices start to rocket once inventory passes a certain threshold because of the quadratic pricing curve. So farming and selling them early into the flat zone is basically throwing money away. So the system predicts if a hinge is likely to be triggered in this game.

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

MARKET_PARAMS = {
    "WHEAT": {"base": 25, "I0": 10000, "T": 400, "below_func": "sqrt", "below_target": 0.80, "above_func": "log", "above_target": 0.20},
    "CARROT": {"base": 35, "I0": 10000, "T": 450, "below_func": "hinge", "below_target": 1.00, "above_func": "sqrt", "above_target": 0.70},
    "TOMATO": {"base": 60, "I0": 10000, "T": 200, "below_func": "hinge", "below_target": 0.40, "above_func": "sqrt", "above_target": 0.60},
    "STRAWBERRY": {"base": 120, "I0": 10000, "T": 100, "below_func": "sqrt", "below_target": 0.70, "above_func": "linear", "above_target": 1.60},
    "MELON": {"base": 250, "I0": 10000, "T": 300, "below_func": "log", "below_target": 0.20, "above_func": "sq", "above_target": 3.60},
    "EGG": {"base": 50, "I0": 10000, "T": 332, "below_func": "hinge", "below_target": 0.40, "above_func": "log", "above_target": 0.20},
    "MILK": {"base": 160, "I0": 10000, "T": 122, "below_func": "sqrt", "below_target": 0.60, "above_func": "linear", "above_target": 1.60},
    "WOOL": {"base": 200, "I0": 10000, "T": 105, "below_func": "log", "below_target": 0.20, "above_func": "sq", "above_target": 3.20},
    "FERTILIZER": {"base": 100, "I0": 10000, "T": 200, "below_func": "linear", "below_target": 0.40, "above_func": "linear", "above_target": 0.40},
}

def hinge_price(crop, inv):
    P = MARKET_PARAMS[crop]
    I0, T, base, tgt = 10000, P["T"], P["base"], P["below_target"]
    u = (I0 - inv) / T
    return base + base * tgt * (u + 8.0 * max(0.0, u - 1.0) ** 2)

with open("/kaggle/input/datasets/rakhansyah/kaggriculture-trace-results/trace_veg.json") as f:
    trace = json.load(f)

CROPS = {"CARROT": "tab:orange", "TOMATO": "tab:red"}
I0 = 10000

fig, ax = plt.subplots(figsize=(12, 6))

ax.set_xlim(9400, I0)
ax.set_ylim(0, 250)

for crop, color in CROPS.items():
    P = MARKET_PARAMS[crop]; T = P["T"]
    trigger, knee = I0 - 0.35*T, I0 - T
    inv = np.linspace(9400, I0, 400)
    ax.plot(inv, [hinge_price(crop, v) for v in inv], color=color, lw=2,
            label=f"{crop} price curve")
    ax.axvline(trigger, color=color, ls="--", lw=1.2,
               label=f"{crop} hinge trigger ({trigger:.0f})")
    ax.axvline(knee, color=color, ls=":", lw=1.2,
               label=f"{crop} knee ({knee:.0f})")
    ax.axvspan(9400, trigger, color=color, alpha=0.05)

for crop, color in CROPS.items():
    for gi, g in enumerate(trace["games"]):
        xs_on, ys_on = [], []
        for i, d in enumerate(g["per_day"]):
            if crop not in d["inventory"]:
                continue
            if bool(d["hinge_triggered"][crop]):
                xs_on.append(d["inventory"][crop]); ys_on.append(d["prices"][crop])
            if xs_on:
                xs_on, ys_on = xs_on[:1], ys_on[:1]
        ax.scatter(xs_on, ys_on, s=70, c=color, edgecolor="k",
                   linewidths=0.4, alpha=0.85, label=f"{crop} hinge ON" if gi == 0 else None)


for crop, color in CROPS.items():
    for gi, g in enumerate(trace["games"]):
        ev = [e for e in g.get("sales", []) if e.get("crop") == crop]
        if not ev:
            continue
        base_y = [hinge_price(crop, e["inventory"]) for e in ev]
        sales_x = [e["inventory"] for e in ev]
        sales_y = [b + 15 for b in base_y]
        sizes   = [min(40 + 2.2 * e["qty"], 220) for e in ev]
        ax.vlines(sales_x, base_y, sales_y, colors=color, lw=0.8, alpha=0.45, zorder=5)
        ax.scatter(sales_x, sales_y, s=sizes, facecolors="white",
                   edgecolors=color, linewidths=2.2, zorder=10,
                   label=f"{crop} SALES (n={sum(e['qty'] for e in ev)})" if gi == 0 else None)

ax.set_xlabel("market inventory (shared, I0 = 10000)")
ax.set_ylabel("sale price ($)")
ax.set_title("CARROT & TOMATO hinge curve")
ax.legend(fontsize=8); ax.grid(alpha=0.3); plt.tight_layout()
plt.show()

The solid dot marks the day when the hinge prediction was triggered. While active the agent would try to hold as much carrots and tomatoes they harvested until the market goes scarce and the price rockets. The white bubbles over the curve is where the agent sells these items. Most of the items were sold on the last day: carrot at $94, and tomato at $148. 

The trigger sits above the knee because of lead time, it checks the demand per day of the item and predicts if the knee is going to be hit over the course of the game. Price only explodes below the knee (I0 minus T), but a tomato takes about 12 days to yield. The agent starts holding at the trigger (I0 minus 0.35T) so that by harvest the inventory has already crashed below the knee. Sell into the flat zone and you throw coins away.

Because it uses the same prediction system, this would directly influence the share ratio in the previous section.

### But tomato's base profit is terrible. Why tomatooo????

In [ ]:
import math


def _shape(func, value, T=None):
    if func == "linear": return value
    if func == "sqrt":   return math.sqrt(value)
    if func == "log":    return math.log1p(value)
    if func == "sq":     return value * value
    if func == "log10":  return math.log10(1.0 + value)
    if func == "hinge":
        u = value / T if T else value
        return u + 8.0 * max(0.0, u - 1.0) ** 2

def price_at(crop, inv):
    P = MARKET_PARAMS[crop]; I0, base, T = 10000, P["base"], P["T"]
    if inv < I0:
        f = _shape(P["below_func"], I0 - inv, T)
        amp = P["below_target"] * base / _shape(P["below_func"], T, T)
        return max(1, round(base + amp * f))
    if inv > I0:
        f = _shape(P["above_func"], inv - I0, T)
        amp = P["above_target"] * base / _shape(P["above_func"], T, T)
        return max(1, round(base - amp * f))
    return base

SEED  = {"WHEAT": 10, "CARROT": 20, "TOMATO": 50, "STRAWBERRY": 100, "MELON": 80}
YIELD = {"WHEAT": 4,  "CARROT": 3,  "TOMATO": 4,  "STRAWBERRY": 4,   "MELON": 6}
CYCLE = {"WHEAT": 5,  "CARROT": 4,  "TOMATO": 12, "STRAWBERRY": 17,  "MELON": 11}

def profit_per_tileday(crop, price):
    return (YIELD[crop] * price - SEED[crop]) / CYCLE[crop]

scenarios = [10000, 9800, 9500]
rows = []
for c in ["WHEAT", "CARROT", "TOMATO", "STRAWBERRY", "MELON"]:
    row = {"crop": c, "seed": SEED[c], "yield": YIELD[c], "cycle": CYCLE[c]}
    for inv in scenarios:
        row[f"price@{inv}"] = price_at(c, inv)
        row[f"profit@{inv}"] = round(profit_per_tileday(c, price_at(c, inv)), 1)
    rows.append(row)
profit_df = pd.DataFrame(rows)

order = profit_df.sort_values("profit@10000", ascending=False)["crop"].tolist()
fig, ax = plt.subplots(figsize=(9, 5.5))
y = np.arange(len(order)); h = 0.36
ax.barh(y + h/2, profit_df.set_index("crop").loc[order, "profit@10000"], height=h,
        color="#8e9aaf", label="base (inv 10000)")
ax.barh(y, profit_df.set_index("crop").loc[order, "profit@9800"], height=h,
        color="#e9c46a", label="mild scarcity (inv 9800)")
ax.barh(y - h/2, profit_df.set_index("crop").loc[order, "profit@9500"], height=h,
        color="#2a9d8f", label="deep scarcity (inv 9500)")
ax.set_yticks(y, order); ax.set_xlabel("profit per tile-day (coins)")
ax.set_title("profit per tile-day by crop — base vs deep scarcity")
ax.legend(); ax.grid(axis="x", alpha=0.3); plt.tight_layout(); plt.show()


Tomato is the worst crop in the game at base price: 15.8 coins per tile-day, half of strawberry's 22.4. You plant it as a bet that the market turns scarce before you harvest. However, when inventory drops below the knee, tomato profit climbs to about 180 per tile-day, eight times strawberry. I've even seen games where the prices reach well over $500.

Well, the number is just an upper bound. It ignores watering and fertilizer labor, and it ignores your own price impact. Dump a big tomato lot and inventory rebounds, which collapses the rocket. That’s why the price prediction also considers the impact of both my own sales and the opponent’s. If the opponent has little to no tomato planted? Hold everything until the last possible moment.

The rest of the chart is plain. Carrot beats wheat at every inventory level, so wheat's only real job is feeding animals. Strawberry earns steadily. Melon dominates the board (129 to 159 per tile-day), which is why the agent prioritize it and races it early-game.

## The Showdown

After banging my head against this rewrite, I wanted proof it was worth it. So I dragged each version back: the greedy, the worker, and the predictor. I ran each against the same opponent (Read the Market, Choose the Farm), on the same seeds, from both seats. The only thing that changed between eras was the agent.

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt

with open("/kaggle/input/datasets/rakhansyah/kaggriculture-trace-results/results.json") as f:
    res = json.load(f)

df = pd.DataFrame(res["games"])
order = ["market_system", "queue_rewrite", "v1_greedy"]
summary = df.groupby("era").agg(
    mean=("margin", "mean"), median=("margin", "median"), std=("margin", "std"),
    wins=("win", "sum"), n=("margin", "size"),
)
summary["win_rate"] = summary["wins"] / summary["n"]

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["#c0392b", "#e67e22", "#27ae60"]
df.boxplot(column="margin", by="era", ax=ax, positions=range(3),
           return_type="axes")
fig.suptitle("")
ax.scatter(range(3), summary.loc[order, "mean"], color=colors, s=80, zorder=3,
           label="mean margin", )
means = summary.loc[order, "mean"]

for i in enumerate(means):
    ax.annotate(f"{i[1]:.0f}", (i[0], i[1]), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9, fontweight="bold", color=colors[i[0]])

ax.axhline(0, color="k", lw=0.8)
ax.set_title("Margin vs Frontier per era")
ax.set_ylabel("final money − frontier money")
plt.tight_layout()

Each rewrite cut the deficit roughly in half: v1 lost by 81k, the rewrite by 64k, the market system by 30k. The spread shrank too, from a standard deviation of 26k to 12k. The agent got faster and more consistent.

Zero wins in six games.

The frontier agent still reads the market and we still trail it, but the gap closes at a measurable rate. But as of now, every change I try to make changes nothing.

---

## The Result

<u>So after all this, what did I learn?</u>

Treat my numbers as a trend, not a leaderboard attempt, as of now. The first version stumbled around doing things and lost. The next version made it lose less. The final version further closed the gap, losing by roughly a third as much as the original.

<u>What's next?</u>

The final target is, well, obvious: become competitive on the leaderboard. The current agent got a top score of 1000. And the hinge chart and plot share ratio made my next target obvious: 

- Soft-locking. The agent predicts and chooses correctly, but gets stuck with the previous crop mix. Seeds are bought correctly, but they can’t be planted until the previous crops are cleared, so the new target ratio doesn’t actually take effect until the plot opens up.

- Greed. when a hinge is predicted it tries to hold as much units as possible of the crop. Sometimes this causes other produce to overflow instead.

- Split and stagger sales. Sell only enough to keep the price climbing, and reduce the risk of an overflow.

With all the bits resolved, I do believe this could be a competitive AI agent. But until then, it's time to go back to the drawing board.